In [ ]:
!git clone https://github.com/pranceraz/DeepHybrid.git
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Google Drive mount skipped: not running in Colab.')

os.chdir('/content/DeepHybrid')
!pwd
!pip install rl4co[graph] torch-geometric

In [ ]:
# ===============================
# COLAB READY DEEPACO TRAINING
# ===============================

import os
from datetime import datetime

import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from generator import MyJSSPGenerator
from my_env import OperationSelectionEnv
from init_embedding import JSSPInitEmbedding, JsspEdgeEmbedding
from aco_class import MyAntSystem

from rl4co.models.zoo.nargnn.encoder import NARGNNEncoder
from rl4co.models.zoo.deepaco.policy import DeepACOPolicy
from rl4co.models.zoo.deepaco.model import DeepACO
from rl4co.models.rl.common.base import RL4COLitModule
from rl4co.utils.trainer import RL4COTrainer


In [ ]:

# ===============================
# CONFIG
# ===============================

NUM_JOBS = 6
NUM_MACHINES = 6
EMBED_DIM = 256

BATCH_SIZE = 64
VAL_BATCH_SIZE = 64

TRAIN_EPOCHS = 30
TRAIN_DATA_SIZE = 8192
VAL_DATA_SIZE = 1024
LR = 1e-4

DRIVE_ROOT = "/content/drive/MyDrive/DeepHybrid"
RUN_NAME = f"deepaco_jssp_{NUM_JOBS}x{NUM_MACHINES}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = os.path.join(DRIVE_ROOT, "training_runs", RUN_NAME)
CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
LIGHTNING_LOG_DIR = os.path.join(RUN_DIR, "lightning_logs")
STATE_DICT_PATH = os.path.join(RUN_DIR, f"{RUN_NAME}.pt")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LIGHTNING_LOG_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)
print("Lightning logs will be saved to:", LIGHTNING_LOG_DIR)


# ===============================
# ENV + GENERATOR
# ===============================

# generator = MyJSSPGenerator(
#     num_jobs=NUM_JOBS,
#     num_machines=NUM_MACHINES
# )


class FixedBatchGenerator:
    def __init__(self, base_generator, batch_size):
        self.base = base_generator
        self.batch_size = batch_size

        # forward attributes
        self.num_jobs = base_generator.num_jobs
        self.num_mas = base_generator.num_mas
        self.n_ops_max = base_generator.n_ops_max

    def __call__(self, batch_size=None):
        # Important: honor requested batch size for RL4CO dataset generation.
        # If caller does not provide one, fall back to default training batch size.
        if batch_size is None:
            batch_size = [self.batch_size]
        elif isinstance(batch_size, int):
            batch_size = [batch_size]
        return self.base(batch_size=batch_size)


base_generator = MyJSSPGenerator(
    num_jobs=NUM_JOBS,
    num_machines=NUM_MACHINES
)

generator = FixedBatchGenerator(base_generator, BATCH_SIZE)




env = OperationSelectionEnv(generator)


# ===============================
# ENCODER
# ===============================

init_emb = JSSPInitEmbedding(
    embed_dim=EMBED_DIM,
    num_machines=NUM_MACHINES
)

edge_emb = JsspEdgeEmbedding(embed_dim=EMBED_DIM)

encoder = NARGNNEncoder(
    embed_dim=EMBED_DIM,
    init_embedding=init_emb,
    edge_embedding=edge_emb
)


# ===============================
# POLICY (DeepACO)
# ===============================

policy = DeepACOPolicy(
    encoder=encoder,
    env_name="tsp",  # symbolic, fine
    n_ants=dict(train=10, val=20, test=50),
    n_iterations=dict(train=1, val=5, test=10),
    aco_class=MyAntSystem
)


# ===============================
# MODEL
# ===============================




model = DeepACO(
    env=env,
    policy=policy,
    train_with_local_search=False,
    batch_size=BATCH_SIZE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=VAL_BATCH_SIZE,
    dataloader_num_workers=4,
    train_data_size=TRAIN_DATA_SIZE,
    val_data_size=VAL_DATA_SIZE,
    optimizer_kwargs=dict(lr=LR),
)


td = env.reset(batch_size=[BATCH_SIZE])
print("Batch size check:", td.batch_size)

checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="deepaco-{epoch:02d}-{step:06d}",
    save_top_k=-1,
    every_n_epochs=1,
    save_last=True,
)

tb_logger = TensorBoardLogger(save_dir=LIGHTNING_LOG_DIR, name="tensorboard")
csv_logger = CSVLogger(save_dir=LIGHTNING_LOG_DIR, name="csv")

# ===============================
# TRAINER
# ===============================

trainer = RL4COTrainer(
    max_epochs=TRAIN_EPOCHS,
    accelerator="gpu" if DEVICE == "cuda" else "cpu",
    devices=1,
    log_every_n_steps=1,
    default_root_dir=RUN_DIR,
    callbacks=[checkpoint_callback],
    logger=[tb_logger, csv_logger],
)

trainer.fit(model)

# ===============================
# SAVE
# ===============================

torch.save(model.state_dict(), STATE_DICT_PATH)
print("Final state dict saved to:", STATE_DICT_PATH)
print("Best checkpoint path:", checkpoint_callback.best_model_path)
print("Last checkpoint path:", checkpoint_callback.last_model_path)

In [ ]:
torch.save(model.state_dict(), STATE_DICT_PATH)
print("Final state dict saved to:", STATE_DICT_PATH)